*Import Library*

In [1]:
import gymnasium as gym
import numpy as np
import metaworld
import random
import os
os.environ['MUJOCO_GL']='osmesa'
import gymnasium as gym
from gymnasium.wrappers import ResizeObservation
from PIL import Image
import matplotlib.pyplot as plt
import importlib
import imageio
import time
from IPython.display import HTML
from base64 import b64encode


In [2]:
# Helper function to get the policy class for a given environment
def get_policy_class(env_name):
    # Map environment names to their corresponding policy class names
    env_to_policy = {
        'reach-v2': 'SawyerReachV2Policy',
        'push-v2': 'SawyerPushV2Policy',
        'pick-place-v2': 'SawyerPickPlaceV2Policy',
        'door-open-v2': 'SawyerDoorOpenV2Policy',
        'drawer-close-v2': 'SawyerDrawerCloseV2Policy',
        'drawer-open-v2': 'SawyerDrawerOpenV2Policy',
        'button-press-topdown-v2': 'SawyerButtonPressTopdownV2Policy',
        'window-open-v2': 'SawyerWindowOpenV2Policy',
        'window-close-v2': 'SawyerWindowCloseV2Policy',
        'peg-insert-side-v2': 'SawyerPegInsertionSideV2Policy'
    }
    
    if env_name not in env_to_policy:
        raise ValueError(f"No policy found for environment {env_name}")
    
    # Convert environment name to import path format
    policy_name = env_to_policy[env_name]
    # Import the policy class
    module = importlib.import_module(f"metaworld.policies.sawyer_{env_name.replace('-', '_')}_policy")
    return getattr(module, policy_name)

In [3]:
from metaworld_env import RandomizeInitialPositionWrapper

In [4]:
# Example usage:
env_name = 'reach-v2'
mt10 = metaworld.MT10(42)
env = mt10.train_classes[env_name](render_mode="rgb_array", camera_name="corner")
env_task_indices = [i for i, task in enumerate(mt10.train_tasks) if task.env_name == env_name]
# print(mt10.train_tasks)
task = mt10.train_tasks[random.choice(env_task_indices)]
env.set_task(task)

# first randomize positions
env = RandomizeInitialPositionWrapper(env)

policy = get_policy_class(env_name)()

In [ ]:
class MakeGoalObservableWrapper(gym.Wrapper):
    """A wrapper that makes the goal position observable in the environment observation."""
    
    def __init__(self, env):
        super().__init__(env)
        # Get the actual SawyerXYZEnv instance
        if hasattr(self.env, 'env'):
            sawyer_env = self.env.env
        else:
            sawyer_env = self.env
            
        # Make goal observable by setting partially_observable to False
        sawyer_env._partially_observable = False
        
        # Update observation space to reflect that goals are now visible
        # This is important if your RL algorithm checks the observation space bounds
        obs_space = sawyer_env.observation_space
        
        # Update the goal portion of observation space (last 3 elements) if needed
        if sawyer_env.goal_space is not None:
            high = obs_space.high.copy()
            low = obs_space.low.copy()
            
            # Set the goal bounds (last 3 elements)
            high[-3:] = sawyer_env.goal_space.high
            low[-3:] = sawyer_env.goal_space.low
            
            # Create updated observation space
            self.observation_space = gym.spaces.Box(
                low=low, 
                high=high,
                dtype=obs_space.dtype
            )

In [10]:
import imageio
# Set up the environment with both wrappers
env_name = 'push-v2'
env = mt10.train_classes[env_name](render_mode="rgb_array", camera_name="corner")
env_task_indices = [i for i, task in enumerate(mt10.train_tasks) if task.env_name == env_name]
task = mt10.train_tasks[random.choice(env_task_indices)]
env.set_task(task)

# Apply wrappers
env = RandomizeInitialPositionWrapper(env)
# env = MakeGoalObservableWrapper(env)

# Create policy
policy = get_policy_class(env_name)()

# Run an episode with the expert policy and save frames
def run_expert_policy_episode(env, policy, max_steps=200):
    frames = []
    obs, _ = env.reset()
    done = False
    step = 0
    total_reward = 0
    
    while not done and step < max_steps:
        # Get action from the expert policy
        action = policy.get_action(obs)
        
        # Execute action
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        
        # Render and save frame
        frame = env.render()
        # Flip the frame vertically to correct the orientation
        frame = np.flipud(frame)
        frames.append(frame)
        
        step += 1
    
    print(f"Episode finished after {step} steps. Total reward: {total_reward}")
    return frames

# Run episode and collect frames
print(f"Running expert policy for {env_name}...")
frames = run_expert_policy_episode(env, policy)

# Save video
video_path = f"expert_policy_{env_name}.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"Video saved to {video_path}")



Running expert policy for push-v2...
Episode finished after 200 steps. Total reward: 1207.2815056033587
Video saved to expert_policy_push-v2.mp4
